# PDDL Attack Path Evaluation
Evaluate generated PDDL attack paths: solvability, syntax, and semantic quality.

## 1. Environment Setup (imports, constnats, global variables)

In [1]:
import sys
import os
import re
from dataclasses import dataclass
from pathlib import Path

from cve2pddlap.core.data_loader import load_few_shot_pool

import torch
from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

from jinja2 import Environment, FileSystemLoader


# Project root (relative — works from notebooks/attack_paths/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# Data paths and file names
DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
GENERATED_DOMAIN_DIR = os.path.join(PROJECT_ROOT, 'generated_domain')
PROMPTS_PATH = os.path.join(PROJECT_ROOT, 'resources', 'prompt', 'evaluation')
AP_PATTERN = re.compile(r'^AP\d+$')
DESCRIPTION_FILE = 'description.txt'
DOMAIN_FILE = 'domain.pddl'
PROBLEM_FILE = 'problem.pddl'

# Models (small defaults — replace with preferred models)
LLM_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Generation parameters
SEED = 42
TEMPERATURE = 0.0
TOP_K = 1


@dataclass(frozen=True)
class EvaluationFlags:
    syntax_check: bool = True
    embedding_intrinsic: bool = True
    embedding_extrinsic: bool = True
    llm_intrinsic: bool = True
    llm_extrinsic: bool = True


eval_flags = EvaluationFlags()

## 2. Data, Prompts, LLM (tokenizer and embedding model)

In [2]:
from cve2pddlap.core.data_loader import load_few_shot_pool

few_shot_pool = load_few_shot_pool(DATASET_PATH)
print(f'Reference examples: {len(few_shot_pool)}')
print(f'Generated domain dir: {os.path.relpath(GENERATED_DOMAIN_DIR)}')
for ex in few_shot_pool[:5]:
    print(f'  {ex.key}')
print('  ...')

Reference examples: 55
Generated domain dir: ../../generated_domain
  CVE-2022-1471 / AP1
  CVE-2022-40149 / AP1
  CVE-2022-40149 / AP2
  CVE-2022-40150 / AP1
  CVE-2022-40150 / AP2
  ...


In [4]:
def load_dataset(data_path):
    """Load all CVEs with their descriptions and attack path PDDL files."""
    dataset = []
    for cve_dir in sorted(Path(data_path).iterdir()):
        if not cve_dir.is_dir():
            continue
        desc_file = cve_dir / DESCRIPTION_FILE
        if not desc_file.exists():
            continue
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir() or not AP_PATTERN.match(ap_dir.name):
                continue
            domain_file = ap_dir / DOMAIN_FILE
            problem_file = ap_dir / PROBLEM_FILE
            if domain_file.exists() and problem_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                    'problem': problem_file.read_text(encoding='utf-8').strip(),
                })
        dataset.append({
            'cve_id': cve_dir.name,
            'description': desc_file.read_text(encoding='utf-8').strip(),
            'attack_paths': attack_paths,
        })
    return dataset


def load_prompts(prompts_path):
    """Load Jinja2 evaluation prompt templates."""
    return Environment(loader=FileSystemLoader(prompts_path))


def load_embedding_model(model_name):
    """Load a SentenceTransformer bi-encoder model."""
    return SentenceTransformer(model_name)


def load_llm(model_name):
    """Load a HuggingFace causal LLM with its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,
        device_map='auto',
    )
    gen = hf_pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=False,
    )
    return gen, tokenizer


dataset = load_dataset(DATASET_PATH)
prompt_env = load_prompts(PROMPTS_PATH)

embedding_model = None
if eval_flags.embedding_intrinsic or eval_flags.embedding_extrinsic:
    embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

llm, tokenizer = None, None
if eval_flags.llm_intrinsic or eval_flags.llm_extrinsic:
    llm, tokenizer = load_llm(LLM_MODEL_NAME)

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## 3. Syntactic and Semantic Evaluation
### 3.1 Syntactic
#### 3.1.1 Intrinsic

##### 

4. Evaluate Domain PDDL (Generated + Reference)
Select domain.pddl files to evaluate. The problem.pddl is **auto-generated** from the domain using code (no LLM).
Then verify solvability (Metric-FF) and syntax (ENHSP).

**Sources:**
- `generated_domain/` — LLM-generated domains
- `resources/data/CVE-PDDL-NNL-ReAP/` — reference domains (55 APs)

In [5]:
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem

ff = create_ff_checker()
enhsp = create_enhsp_checker()

# --- Build domain list: generated + reference ---
def _scan_domains():
    domains = []
    for f in sorted(glob.glob(os.path.join(GENERATED_DOMAIN_DIR, '*.pddl'))):
        if not f.endswith('_problem.pddl'):
            label = f'[generated] {os.path.basename(f)}'
            domains.append((label, f))
    for f in sorted(glob.glob(os.path.join(DATASET_PATH, '*/AP*/domain.pddl'))):
        parts = f.split('/')
        key = f'{parts[-3]}/{parts[-2]}'
        label = f'[reference] {key}'
        domains.append((label, f))
    return domains

domain_options = _scan_domains()

domain_select = widgets.SelectMultiple(
    options=domain_options,
    description='Domains:',
    rows=15,
    layout=widgets.Layout(width='600px'),
    style={'description_width': '70px'},
)

select_all_gen_btn = widgets.Button(description='Select all generated', layout=widgets.Layout(width='180px'))
select_all_ref_btn = widgets.Button(description='Select all reference', layout=widgets.Layout(width='180px'))
select_all_btn = widgets.Button(description='Select all', layout=widgets.Layout(width='120px'))
clear_btn = widgets.Button(description='Clear', layout=widgets.Layout(width='80px'))
refresh_btn = widgets.Button(description='Refresh list', button_style='warning', layout=widgets.Layout(width='120px'))

def _select_all_gen(b):
    domain_select.value = [v for l, v in domain_options if l.startswith('[generated]')]
def _select_all_ref(b):
    domain_select.value = [v for l, v in domain_options if l.startswith('[reference]')]
def _select_all(b):
    domain_select.value = [v for _, v in domain_options]
def _clear(b):
    domain_select.value = []
def _refresh(b):
    global domain_options
    domain_options = _scan_domains()
    domain_select.options = domain_options
    print(f'Refreshed: {len(domain_options)} domains found')

select_all_gen_btn.on_click(_select_all_gen)
select_all_ref_btn.on_click(_select_all_ref)
select_all_btn.on_click(_select_all)
clear_btn.on_click(_clear)
refresh_btn.on_click(_refresh)

display(widgets.VBox([
    widgets.HTML('<b>Select domains to evaluate</b> (Ctrl/Cmd+click for multi-select)'),
    widgets.HBox([select_all_gen_btn, select_all_ref_btn, select_all_btn, clear_btn, refresh_btn]),
    domain_select,
]))
print(f'Total available: {len(domain_options)} domains')

Total available: 103 domains


In [ ]:
selected_domains = list(domain_select.value)
if not selected_domains:
    raise ValueError('No domains selected. Run the cell above and select domains first.')

print(f'Generating problem.pddl for {len(selected_domains)} domain(s)...\n')

eval_items = []

for domain_path in selected_domains:
    if GENERATED_DOMAIN_DIR in domain_path:
        source = 'generated'
        display_name = os.path.basename(domain_path).replace('.pddl', '')
    else:
        parts = domain_path.split('/')
        source = 'reference'
        display_name = f'{parts[-3]}/{parts[-2]}'

    try:
        with open(domain_path) as f:
            domain_pddl = f.read()
        problem_pddl = generate_problem(domain_pddl)
        eval_items.append((display_name, source, domain_path, problem_pddl))
        print(f'  \u2713 {display_name} [{source}]')
    except Exception as e:
        print(f'  \u2717 {display_name} [{source}] — ERROR: {e}')

print(f'\nProblem generation done: {len(eval_items)}/{len(selected_domains)} succeeded')

In [ ]:
# Step 2 (optional): Preview generated problem.pddl
# Change idx to inspect different domains

idx = 0  # change this

if eval_items:
    name, source, dpath, problem = eval_items[idx]
    print(f'=== {name} [{source}] ===')
    print(f'Domain: {dpath}\n')
    print(problem)
else:
    print('No items. Run the cell above first.')

=== CVE-2025-66032_qwen-max_1shot [generated] ===
Domain: /Users/cuilin/claudework/llm-ap-generation/llm-ap-generation/experiments/CVE-2025-66032_qwen-max_1shot.pddl

(define (problem AEDI-elevation-of-privilege)
  (:domain AED)
  (:objects
    arbitrary-code_SEFA - arbitrary-code
    coding-tool_SEFA - coding-tool
    context-window_SEFA - context-window
    SEFA - target-system
    untrusted-content_SEFA - untrusted-content
    validation-bypass_SEFA - validation-bypass
  )
  (:init
    (= (total-cost) 0)
    (has-coding-tool SEFA coding-tool_SEFA)
    (context-window-exists SEFA context-window_SEFA)
    (shell-command-parsing-error coding-tool_SEFA)
    (ifs-variable-exploitable coding-tool_SEFA)
    (short-cli-flag-exploitable coding-tool_SEFA)
    (= (version coding-tool_SEFA) 100092000)
  )
  (:goal (and (elevation-of-privilege SEFA)))
  (:metric minimize (total-cost)))



In [ ]:
# Step 3: Evaluate solvability (Metric-FF) and syntax (ENHSP)

import tempfile

eval_results = []
print(f'Running FF + ENHSP on {len(eval_items)} domain(s)...\n')

for display_name, source, domain_path, problem_pddl in eval_items:
    # Write problem to temp file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.pddl', delete=False) as tmp:
        tmp.write(problem_pddl)
        tmp_path = tmp.name

    r_ff = ff.check(domain_path, tmp_path)
    r_enhsp = enhsp.check(domain_path, tmp_path)
    os.unlink(tmp_path)

    ff_ok = '\u2713' if r_ff.solvable else '\u2717'
    enhsp_ok = '\u2713' if r_enhsp.success else '\u2717'

    eval_results.append({
        'name': display_name,
        'source': source,
        'ff_solvable': r_ff.solvable,
        'ff_plan_length': r_ff.plan_length,
        'ff_cost': r_ff.plan_cost,
        'ff_error': r_ff.error,
        'ff_plan': r_ff.plan,
        'enhsp_ok': r_enhsp.success,
        'enhsp_error': r_enhsp.error,
    })

    print(f'  {display_name} [{source}]')
    print(f'    FF: {ff_ok}  len={r_ff.plan_length}  cost={r_ff.plan_cost}')
    if r_ff.error:
        print(f'    FF error: {r_ff.error[:200]}')
    print(f'    ENHSP: {enhsp_ok}')
    if r_enhsp.error:
        print(f'    ENHSP error: {r_enhsp.error[:200]}')
    if r_ff.plan:
        print(f'    Plan ({len(r_ff.plan)} steps):')
        for i, a in enumerate(r_ff.plan):
            print(f'      {i}: {a}')
    print()

# Summary
n = len(eval_results)
n_ff = sum(1 for r in eval_results if r['ff_solvable'])
n_enhsp = sum(1 for r in eval_results if r['enhsp_ok'])
print(f'=== Summary ===')
print(f'Total: {n} | FF solvable: {n_ff}/{n} | ENHSP syntax OK: {n_enhsp}/{n}')

Running FF + ENHSP on 1 domain(s)...

  CVE-2025-66032_qwen-max_1shot [generated]
    FF: ✓  len=4  cost=33.0
    ENHSP: ✓
    Plan (4 steps):
      0: ATTACKER-INJECTS-UNTRUSTED-CONTENT-INTO-CONTEXT-WINDOW SEFA UNTRUSTED-CONTENT_SEFA CONTEXT-WINDOW_SEFA
      1: ATTACKER-MANIPULATES-CLI-FLAGS SEFA UNTRUSTED-CONTENT_SEFA CLI-FLAG_SEFA
      2: ATTACKER-INJECTS-SHELL-COMMAND SEFA CLI-FLAG_SEFA SHELL-COMMAND_SEFA
      3: TARGET-SYSTEM-EXECUTES-ARBITRARY-CODE SEFA SHELL-COMMAND_SEFA

=== Summary ===
Total: 1 | FF solvable: 1/1 | ENHSP syntax OK: 1/1


In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem

ff = create_ff_checker()
enhsp = create_enhsp_checker()

def _scan_domains():
    domains = []
    for f in sorted(glob.glob(os.path.join(GENERATED_DOMAIN_DIR, '*.pddl'))):
        if not f.endswith('_problem.pddl'):
            label = f'[generated] {os.path.basename(f)}'
            domains.append((label, f))
    for f in sorted(glob.glob(os.path.join(DATASET_PATH, '*/AP*/domain.pddl'))):
        parts = f.split('/')
        key = f'{parts[-3]}/{parts[-2]}'
        label = f'[reference] {key}'
        domains.append((label, f))
    return domains

domain_options = _scan_domains()

domain_select = widgets.SelectMultiple(
    options=domain_options,
    description='Domains:',
    rows=15,
    layout=widgets.Layout(width='600px'),
    style={'description_width': '70px'},
)

select_all_gen_btn = widgets.Button(description='Select all generated', layout=widgets.Layout(width='180px'))
select_all_ref_btn = widgets.Button(description='Select all reference', layout=widgets.Layout(width='180px'))
select_all_btn = widgets.Button(description='Select all', layout=widgets.Layout(width='120px'))
clear_btn = widgets.Button(description='Clear', layout=widgets.Layout(width='80px'))
refresh_btn = widgets.Button(description='Refresh list', button_style='warning', layout=widgets.Layout(width='120px'))

def _select_all_gen(b):
    domain_select.value = [v for l, v in domain_options if l.startswith('[generated]')]
def _select_all_ref(b):
    domain_select.value = [v for l, v in domain_options if l.startswith('[reference]')]
def _select_all(b):
    domain_select.value = [v for _, v in domain_options]
def _clear(b):
    domain_select.value = []
def _refresh(b):
    global domain_options
    domain_options = _scan_domains()
    domain_select.options = domain_options
    print(f'Refreshed: {len(domain_options)} domains found')

select_all_gen_btn.on_click(_select_all_gen)
select_all_ref_btn.on_click(_select_all_ref)
select_all_btn.on_click(_select_all)
clear_btn.on_click(_clear)
refresh_btn.on_click(_refresh)

display(widgets.VBox([
    widgets.HTML('<b>Select domains to evaluate</b> (Ctrl/Cmd+click for multi-select)'),
    widgets.HBox([select_all_gen_btn, select_all_ref_btn, select_all_btn, clear_btn, refresh_btn]),
    domain_select,
]))
print(f'Total available: {len(domain_options)} domains')

In [ ]:
selected_domains = list(domain_select.value)
if not selected_domains:
    raise ValueError('No domains selected. Run the cell above and select domains first.')

print(f'Generating problem.pddl for {len(selected_domains)} domain(s)...\n')

eval_items = []

for domain_path in selected_domains:
    if GENERATED_DOMAIN_DIR in domain_path:
        source = 'generated'
        display_name = os.path.basename(domain_path).replace('.pddl', '')
    else:
        parts = domain_path.split('/')
        source = 'reference'
        display_name = f'{parts[-3]}/{parts[-2]}'

    try:
        with open(domain_path) as f:
            domain_pddl = f.read()
        problem_pddl = generate_problem(domain_pddl)
        eval_items.append((display_name, source, domain_path, problem_pddl))
        print(f'  \u2713 {display_name} [{source}]')
    except Exception as e:
        print(f'  \u2717 {display_name} [{source}] — ERROR: {e}')

print(f'\nProblem generation done: {len(eval_items)}/{len(selected_domains)} succeeded')

In [ ]:
# Step 2 (optional): Preview generated problem.pddl
# Change idx to inspect different domains

idx = 0  # change this

if eval_items:
    name, source, dpath, problem = eval_items[idx]
    print(f'=== {name} [{source}] ===')
    print(f'Domain: {dpath}\n')
    print(problem)
else:
    print('No items. Run the cell above first.')

=== CVE-2025-66032_gpt4_1shot [generated] ===
Domain: /Users/cuilin/claudework/llm-ap-generation/llm-ap-generation/experiments/CVE-2025-66032_gpt4_1shot.pddl

(define (problem AEDI-elevation-of-privilege)
  (:domain AED)
  (:objects
    coding-tool_SEFA - coding-tool
    context-window_SEFA - context-window
    SEFA - target-system
    untrusted-content_SEFA - untrusted-content
  )
  (:init
    (= (total-cost) 0)
    (has-coding-tool SEFA coding-tool_SEFA)
    (vulnerable-claude-code coding-tool_SEFA)
    (context-window-accepts-untrusted-content coding-tool_SEFA context-window_SEFA)
    (= (version coding-tool_SEFA) 1009199000)
  )
  (:goal (and (elevation-of-privilege SEFA)))
  (:metric minimize (total-cost)))



In [ ]:
# Step 3: Evaluate solvability (Metric-FF) and syntax (ENHSP)

import tempfile

eval_results = []
print(f'Running FF + ENHSP on {len(eval_items)} domain(s)...\n')

for display_name, source, domain_path, problem_pddl in eval_items:
    # Write problem to temp file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.pddl', delete=False) as tmp:
        tmp.write(problem_pddl)
        tmp_path = tmp.name

    r_ff = ff.check(domain_path, tmp_path)
    r_enhsp = enhsp.check(domain_path, tmp_path)
    os.unlink(tmp_path)

    ff_ok = '\u2713' if r_ff.solvable else '\u2717'
    enhsp_ok = '\u2713' if r_enhsp.success else '\u2717'

    eval_results.append({
        'name': display_name,
        'source': source,
        'ff_solvable': r_ff.solvable,
        'ff_plan_length': r_ff.plan_length,
        'ff_cost': r_ff.plan_cost,
        'ff_error': r_ff.error,
        'ff_plan': r_ff.plan,
        'enhsp_ok': r_enhsp.success,
        'enhsp_error': r_enhsp.error,
    })

    print(f'  {display_name} [{source}]')
    print(f'    FF: {ff_ok}  len={r_ff.plan_length}  cost={r_ff.plan_cost}')
    if r_ff.error:
        print(f'    FF error: {r_ff.error[:200]}')
    print(f'    ENHSP: {enhsp_ok}')
    if r_enhsp.error:
        print(f'    ENHSP error: {r_enhsp.error[:200]}')
    if r_ff.plan:
        print(f'    Plan ({len(r_ff.plan)} steps):')
        for i, a in enumerate(r_ff.plan):
            print(f'      {i}: {a}')
    print()

# Summary
n = len(eval_results)
n_ff = sum(1 for r in eval_results if r['ff_solvable'])
n_enhsp = sum(1 for r in eval_results if r['enhsp_ok'])
print(f'=== Summary ===')
print(f'Total: {n} | FF solvable: {n_ff}/{n} | ENHSP syntax OK: {n_enhsp}/{n}')

Running FF + ENHSP on 1 domain(s)...

  CVE-2025-66032_gpt4_1shot [generated]
    FF: ✓  len=3  cost=32.0
    ENHSP: ✓
    Plan (3 steps):
      0: ATTACKER-INJECTS-UNTRUSTED-CONTENT-INTO-CONTEXT-WINDOW SEFA CONTEXT-WINDOW_SEFA UNTRUSTED-CONTENT_SEFA CODING-TOOL_SEFA
      1: TARGET-SYSTEM-TRIGGERS-ARBITRARY-CODE-EXECUTION SEFA CONTEXT-WINDOW_SEFA UNTRUSTED-CONTENT_SEFA
      2: TARGET-SYSTEM-EXPERIENCES-CODE-EXECUTION-VULNERABILITY-IMPACT SEFA

=== Summary ===
Total: 1 | FF solvable: 1/1 | ENHSP syntax OK: 1/1


## 5. Syntactic Extrinsic (future)

In [ ]:
# TODO: syntactic evaluation

## 6. Semantic Evaluation

### 6.1 Intrinsic

#### 6.1.1 Embedding — NL Description vs PDDL Similarity

In [ ]:
def embedding_similarity_intrinsic(description, domain, problem, model):
    """Cosine similarity between NL description and PDDL code (domain + problem)."""
    pddl_text = domain + '\n' + problem
    embeddings = model.encode(
        [description, pddl_text],
        convert_to_tensor=True,
        normalize_embeddings=True,
    )
    return st_util.cos_sim(embeddings[0], embeddings[1]).item()

In [ ]:
emb_intrinsic_results = []
for entry in dataset:
    for ap in entry['attack_paths']:
        sim = embedding_similarity_intrinsic(
            entry['description'], ap['domain'], ap['problem'], embedding_model
        )
        emb_intrinsic_results.append({
            'cve_id': entry['cve_id'],
            'ap_id': ap['ap_id'],
            'similarity': sim,
        })

emb_intrinsic_results

#### 6.1.2 LLM — NL Description vs PDDL Match

In [ ]:
INTRINSIC_LLM_PROMPT = """You are given a CVE vulnerability description and a PDDL domain + problem that models an attack path for that vulnerability.

Evaluate whether the PDDL code correctly models the vulnerability described.
Answer with "YES" if it correctly matches, "NO" if it does not, followed by a brief explanation.

## CVE Description
{description}

## PDDL Domain
{domain}

## PDDL Problem
{problem}

## Evaluation
"""


def llm_eval_intrinsic(description, domain, problem, generator):
    """Ask LLM whether the PDDL code matches the NL description."""
    prompt = INTRINSIC_LLM_PROMPT.format(
        description=description, domain=domain, problem=problem
    )
    messages = [{'role': 'user', 'content': prompt}]
    output = generator(messages)
    return output[0]['generated_text'][-1]['content']

In [ ]:
llm_intrinsic_results = []
for entry in dataset:
    for ap in entry['attack_paths']:
        response = llm_eval_intrinsic(
            entry['description'], ap['domain'], ap['problem'], llm
        )
        llm_intrinsic_results.append({
            'cve_id': entry['cve_id'],
            'ap_id': ap['ap_id'],
            'llm_response': response,
        })

llm_intrinsic_results

### 6.2 Extrinsic

#### 6.2.1 Embedding — PDDL vs PDDL Similarity

In [ ]:
def embedding_similarity_extrinsic(domain_a, problem_a, domain_b, problem_b, model):
    """Cosine similarity between two PDDL code blocks (domain + problem)."""
    pddl_a = domain_a + '\n' + problem_a
    pddl_b = domain_b + '\n' + problem_b
    embeddings = model.encode(
        [pddl_a, pddl_b],
        convert_to_tensor=True,
        normalize_embeddings=True,
    )
    return st_util.cos_sim(embeddings[0], embeddings[1]).item()

In [ ]:
emb_extrinsic_results = []
for entry in dataset:
    aps = entry['attack_paths']
    if len(aps) < 2:
        continue
    for i in range(len(aps)):
        for j in range(i + 1, len(aps)):
            sim = embedding_similarity_extrinsic(
                aps[i]['domain'], aps[i]['problem'],
                aps[j]['domain'], aps[j]['problem'],
                embedding_model,
            )
            emb_extrinsic_results.append({
                'cve_id': entry['cve_id'],
                'ap_a': aps[i]['ap_id'],
                'ap_b': aps[j]['ap_id'],
                'similarity': sim,
            })

emb_extrinsic_results

#### 6.2.2 LLM — Reference PDDL vs Candidate PDDL Match

In [ ]:
EXTRINSIC_LLM_PROMPT = """You are given a CVE vulnerability description, a reference PDDL attack path (domain + problem) that is known to be correct, and a candidate PDDL attack path (domain + problem).

Evaluate whether the candidate PDDL code correctly models the same vulnerability as the reference.
Answer with "YES" if it correctly matches, "NO" if it does not, followed by a brief explanation.

## CVE Description
{description}

## Reference PDDL Domain
{ref_domain}

## Reference PDDL Problem
{ref_problem}

## Candidate PDDL Domain
{cand_domain}

## Candidate PDDL Problem
{cand_problem}

## Evaluation
"""


def llm_eval_extrinsic(description, ref_domain, ref_problem, cand_domain, cand_problem, generator):
    """Ask LLM whether candidate PDDL matches the reference given the NL spec."""
    prompt = EXTRINSIC_LLM_PROMPT.format(
        description=description,
        ref_domain=ref_domain,
        ref_problem=ref_problem,
        cand_domain=cand_domain,
        cand_problem=cand_problem,
    )
    messages = [{'role': 'user', 'content': prompt}]
    output = generator(messages)
    return output[0]['generated_text'][-1]['content']

In [ ]:
llm_extrinsic_results = []
for entry in dataset:
    aps = entry['attack_paths']
    if len(aps) < 2:
        continue
    ref = aps[0]
    for cand in aps[1:]:
        response = llm_eval_extrinsic(
            entry['description'],
            ref['domain'], ref['problem'],
            cand['domain'], cand['problem'],
            llm,
        )
        llm_extrinsic_results.append({
            'cve_id': entry['cve_id'],
            'ref_ap': ref['ap_id'],
            'cand_ap': cand['ap_id'],
            'llm_response': response,
        })

llm_extrinsic_results

## 7. Human Evaluation (future)

In [ ]:
# TODO: human evaluation questionnaire design